# Person 1 (Aarya): run the pipelines on a Colab GPU

**Before running anything:** menu **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is the free one; L4/A100 if you have Colab Pro) → Save.

Then run the cells **from top to bottom** (click a cell, press **Shift + Enter**). Each cell says what it does and what you should see.

If Colab disconnects, just run all cells again from the top: finished questions are skipped, nothing is lost *as long as you saved results* (step 7).

## 1. Check the GPU
You should see a table with a GPU name (Tesla T4, L4 or A100). If it says *command not found*, the runtime has no GPU: redo the menu step above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download the project from GitHub
Re-running this cell later just pulls the newest version of `main`.

In [ ]:
import os
REPO = "/content/LateComers-CSE576-NLP"
if not os.path.exists(REPO):
    !git clone https://github.com/aaryapendharkar/LateComers-CSE576-NLP.git {REPO}
%cd {REPO}
!git pull -q origin main
!git log --oneline -1

## 3. Install the packages (2–4 minutes)
Ignore red *dependency conflict* warnings unless the cell ends with an error.

In [ ]:
!pip install -q -r requirements-core.txt
!pip install -q -U transformers accelerate sentence-transformers
import transformers; print("transformers", transformers.__version__)

## 4. Pick the model that fits this GPU
Writes `configs/local.yaml` (only on this Colab machine; never uploaded).
- ≥ 30 GB GPU (A100): the planned **Qwen3.5-9B**
- smaller GPU (T4 / L4): **Qwen3.5-4B**, same family. **Write down which one was used**: it goes in the report's *Changes* section.

In [ ]:
import torch, yaml
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
big = mem_gb >= 30
local = {
    "pipelines": {"backend": "real", "p1_text_source": "standin"},
    "models": {
        "generator_fallback": None if big else "Qwen/Qwen3.5-4B",
        "reranker": None,
        "device": "cuda",
        "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
        "batch_size": 8 if big else 4,
    },
}
with open("configs/local.yaml", "w") as f:
    yaml.safe_dump(local, f)
model = "Qwen/Qwen3.5-9B" if big else "Qwen/Qwen3.5-4B"
print(f"GPU: {gpu} ({mem_gb:.0f} GB)  ->  generator: {model}, dtype: {local['models']['dtype']}")

## 5. The model test: 5 questions through P1 (text pipeline)
The first run downloads the models (BGE-M3 ≈ 2 GB, Qwen3.5 ≈ 8–19 GB), so it takes **5–15 minutes**. It ends by printing a small table.

In [ ]:
!rm -f results/runs/P1_R2_finqa.jsonl   # start the test fresh
!python scripts/run_baselines.py --pipelines P1 --runs R2 --limit 5

## 6. Look at the answers
**What good looks like:**
- `answer` is a number (or yes/no) for every row, not empty
- `cited` lists page ids
- `gold_rank` is mostly 1–5 (retrieval found the right page)
- `correct` is True for at least some rows (a few wrong answers are normal)

Copy this table into the chat with Claude.

In [ ]:
import pandas as pd, json
rows = [json.loads(l) for l in open("results/runs/P1_R2_finqa.jsonl")]
ex = {json.loads(l)["qid"]: json.loads(l) for l in open("data/examples.jsonl")}
df = pd.DataFrame([{
    "qid": r["qid"], "gold": ex[r["qid"]]["gold_answer"][:10], "answer": r["answer"],
    "correct": r["correct"], "gold_rank": r["gold_rank_retr"], "cited": r["cited"],
    "secs": r["latency_s"], "raw_output_start": r["answer_raw"][:80],
} for r in rows])
pd.set_option("display.max_colwidth", 80)
df

## 7. Save results to your Google Drive
Colab deletes everything when it disconnects. This copies `results/` to **My Drive / cse576_results** (it will ask permission to access Drive the first time).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cse576_results
!cp -r results/. /content/drive/MyDrive/cse576_results/
!ls /content/drive/MyDrive/cse576_results/runs

---
## Later steps (don't run yet)
Friday, after Aaron's renderer and pages are on `main`, Claude will give you the next cells (parse pages, build indexes, full baselines). They go below this line.